Train model in setup stage

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
# 是否有可用 GPU
print("CUDA available:", torch.cuda.is_available())

# 当前 GPU 名称
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA available: True
Current GPU: NVIDIA GeForce RTX 3070 Laptop GPU


In [2]:
# Train basic setup model
from DeepLearning.PPO import MaskablePPO
from DeepLearning.Thesis.Environments.Setup import SetupDiversity
from DeepLearning.Thesis.Environments.Setup import SetupRandom
from DeepLearning.Thesis.Setup.getActionMaskSetup import getSetupActionMask
from DeepLearning.Thesis.Setup.getObservationSetup import getObservationSetup, lowerBound, upperBound
from DeepLearning.Thesis.Environments.ObservationTesting import ObsTestingEnv
import os


env = SetupRandom()
actionMask = getSetupActionMask # 126
observation = getObservationSetup # 940

netArchDict = dict(pi=[128, 128, 128], vf=[128, 128, 128])
gamma = 0.99
n_steps = 400

saveName = "ZKA_Try2_SetupRandom_1M"
savePath = f"DeepLearning/Thesis/Setup/Models/{saveName}"

model = MaskablePPO("MlpPolicy", env, verbose=1, policy_kwargs=dict(net_arch=netArchDict), gamma=gamma, n_steps=n_steps, getActionMask=actionMask, getObservation=observation, savePath=savePath, tensorboard_log="./tensorboard_logs_thesis/")
# model = MaskablePPO.load("DeepLearning/Thesis/Setup/Models/SetupRandom/model_332400_5.zip", env=env)
model.savePath = savePath
model.learn(total_timesteps=3_000_000, tb_log_name=saveName, reset_num_timesteps=False)

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

ImportError: DLL load failed while importing _multiarray_umath: 找不到指定的模块。

D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute '_function_setstate' on <module 'cloudpickle.cloudpickle' from 'D:\\ProgramData\\Anaconda3_\\envs\\catan\\Lib\\site-packages\\cloudpickle\\cloudpickle.py'>
  warnings.warn(
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute '_function_setstate' on <module 'cloudpickle.cloudpickle' from 'D:\\ProgramData\\Anaconda3_\\envs\\catan\\Lib\\site-packages\\cloudpickle\\cloudpickle.py'>
  warnings.warn(
D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\utils.py:168: UserWarning: get_schedule_fn() is deprecated, please use FloatS

ImportError: cannot import name 'getObservation' from 'DeepLearning.Thesis.Observations.get_observation_full' (D:\AAA_Projects\Catan\TC2_ZKA\Client\DeepLearning\Thesis\Observations\get_observation_full.py)

In [2]:
# 保存训练好的模型
model.save(savePath)
print(f"Model saved to {savePath}")

Model saved to DeepLearning/Thesis/Setup/Models/ZKA_SetupRandom_1M/ZKA_SetupRandom_1M.zip


D:\ProgramData\Anaconda3_\envs\catan\Lib\site-packages\stable_baselines3\common\save_util.py:284: UserWarning: Path 'DeepLearning\Thesis\Setup\Models\ZKA_SetupRandom_1M' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


Run Agent simulations

In [6]:
"""
Running Agent simulations
"""
from Agents.AgentRandom2 import AgentRandom2
from Agents.AgentMCTS import AgentMCTS
from Agents.AgentUCTTuned import AgentUCTTuned
from Agents.AgentModel import AgentMultiModel, AgentModel
#from Agents.AgentGlobalModel import AgentGlobalModel
from Game.CatanGame import *
from CatanSimulator import CreateGame
from DeepLearning.PPO import MaskablePPO
from Game.CatanPlayer import PlayerStatsTracker
from tabulate import tabulate
from DeepLearning.Stats import headers
import dill as pickle
from CatanData.GameStateViewer import SaveGameStateImage, DisplayImage
import math
import time

winner = [0,0,0,0]
player0Stats = PlayerStatsTracker()
Player0LosingStats = PlayerStatsTracker()
player1Stats = PlayerStatsTracker()
player2Stats = PlayerStatsTracker()
player3Stats = PlayerStatsTracker()

setupModel = MaskablePPO.load("DeepLearning/Thesis/Setup/Models/ZKA_SetupRandom_1M.zip")


#testModel0 = MaskablePPO.load("DeepLearning/Thesis/6.DenseRewards/Models/SelfPlayDense/model_19701760_125.zip")
'''
def make_players(setupModel):
    return [
        AgentRandom2("P0", 0, recordStats=True, playerTrading=False),
        AgentMultiModel(
            "P1", 1,
            recordStats=True,
            playerTrading=False,
            setupModel=setupModel,
            fullSetup=True,
            model=None
        ),
        AgentRandom2("P2", 2, recordStats=True, playerTrading=False),
        AgentRandom2("P3", 3, recordStats=True, playerTrading=False),
    ]
'''
def make_players(setupModel):
    return [
        AgentMCTS("P0", 0, choiceTime = 0.5, multiThreading = False, trading=False),
        AgentMultiModel(
            "P1", 1,
            recordStats=True,
            playerTrading=False,
            setupModel=setupModel,
            fullSetup=True,
            model=AgentMCTS("P1", 1, choiceTime = 0.5, multiThreading = False, trading=False)
        ),
        AgentRandom2("P2", 2),
        AgentRandom2("P3", 3),
    ]


COLLECT_STATS = True

for episode in range(100):
    players = make_players(setupModel)
    game = CreateGame(players)
    #game = pickle.loads(pickle.dumps(game, -1))
    numTurns = 0
    while True:
        currPlayer = game.gameState.players[game.gameState.currPlayer]
        #print("Current Player: " + str(game.gameState.currPlayer))

        agentAction = currPlayer.DoMove(game)
        agentAction.ApplyAction(game.gameState)
        #print(" Take Action: " + agentAction.type)

        if currPlayer.seatNumber == 1 and agentAction.type == 'EndTurn':
            #DisplayImage(game.gameState, agentAction)
            #time.sleep(1)
            numTurns += 1
            print("Turn: ", numTurns)

        if game.gameState.currState == "OVER":
            break

    print("Winner: ", game.gameState.winner)
    winner[game.gameState.winner] += 1
    lost = game.gameState.winner != 0

    # print(winner)

    # Stats
    if COLLECT_STATS:
        game.gameState.players[0].generatePlayerStats()
        game.gameState.players[1].generatePlayerStats()
        game.gameState.players[2].generatePlayerStats()
        game.gameState.players[3].generatePlayerStats()

        player0Stats += game.gameState.players[0].stats
        player1Stats += game.gameState.players[1].stats
        player2Stats += game.gameState.players[2].stats
        player3Stats += game.gameState.players[3].stats
        if lost:
            Player0LosingStats += game.gameState.players[0].stats

# Collect stats
if COLLECT_STATS:
    player0Stats.getAverages()
    Player0LosingStats.getAverages()
    player1Stats.getAverages()
    player2Stats.getAverages()
    player3Stats.getAverages()
    player0Data = player0Stats.getList()
    player0LosingData = Player0LosingStats.getList()
    player1Data = player1Stats.getList()
    player2Data = player2Stats.getList()
    player3Data = player3Stats.getList()

    p_hat0 = winner[0] / sum(winner)
    p_hat1 = winner[1] / sum(winner)
    p_hat2 = winner[0] / sum(winner)
    p_hat3 = winner[1] / sum(winner)
    margin_error0 = round(100*(1.96 * math.sqrt((p_hat0 * (1 - p_hat0)) / sum(winner))), 2)
    margin_error1 = round(100*(1.96 * math.sqrt((p_hat1 * (1 - p_hat1)) / sum(winner))), 2)
    margin_error2 = round(100*(1.96 * math.sqrt((p_hat0 * (1 - p_hat0)) / sum(winner))), 2)
    margin_error3 = round(100*(1.96 * math.sqrt((p_hat1 * (1 - p_hat1)) / sum(winner))), 2)
player0Data.insert(0, margin_error0)
    player0LosingData.insert(0, -1)
    player1Data.insert(0, margin_error1)
    player2Data.insert(0, margin_error2)
    player3Data.insert(0, margin_error3)
    player0Data.insert(0, winner[0]/sum(winner))
    player0LosingData.insert(0, -1)
    player1Data.insert(0, winner[1]/sum(winner))
    player2Data.insert(0, winner[2]/sum(winner))
    player3Data.insert(0, winner[3]/sum(winner))
    player0Data.insert(0, "Player0")
    player0LosingData.insert(0, "Player0LossesStats")
    player1Data.insert(0, "Player1")
    player2Data.insert(0, "Player2")
    player3Data.insert(0, "Player3")

    table = tabulate([player0Data, player0LosingData, player1Data, player2Data, player3Data], headers=headers, tablefmt='simple')
    print(table)

print(f"\nNum turns: {numTurns}")

print("\n\nWinnings: ", winner)


Turn:  1
Turn:  2
Turn:  3
Turn:  4
Turn:  5
Turn:  6
Turn:  7
Turn:  8
Turn:  9
Turn:  10
Turn:  11
Turn:  12
Turn:  13
Turn:  14
Turn:  15
Turn:  16
Turn:  17
Turn:  18
Turn:  19
Turn:  20
Turn:  21
Turn:  22
Turn:  23
Turn:  24
Turn:  25
Turn:  26
Turn:  27
Turn:  28
Turn:  29
Turn:  30
Turn:  31
Turn:  32
Turn:  33
Turn:  34
Turn:  35
Turn:  36
Turn:  37
Turn:  38
Turn:  39
Turn:  40
Winner:  0
Turn:  1
Turn:  2
Turn:  3
Turn:  4
Turn:  5
Turn:  6
Turn:  7
Turn:  8
Turn:  9
Turn:  10
Turn:  11
Turn:  12
Turn:  13
Turn:  14
Turn:  15
Turn:  16
Turn:  17
Turn:  18
Turn:  19
Turn:  20
Turn:  21
Turn:  22
Turn:  23
Turn:  24
Turn:  25
Turn:  26
Turn:  27
Turn:  28
Turn:  29
Turn:  30
Turn:  31
Turn:  32
Turn:  33
Turn:  34
Turn:  35
Turn:  36
Winner:  1
Turn:  1
Turn:  2
Turn:  3
Turn:  4
Turn:  5
Turn:  6
Turn:  7
Turn:  8
Turn:  9
Turn:  10
Turn:  11
Turn:  12
Turn:  13
Turn:  14
Turn:  15
Turn:  16
Turn:  17
Turn:  18
Turn:  19
Turn:  20
Turn:  21
Turn:  22
Turn:  23
Turn:  24
Turn:

In [3]:
import pandas as pd

# Save to csv
fileName = f'SetupRandom_v_Random.csv'
df = pd.DataFrame([player0Data, player1Data, player2Data, player3Data], columns=headers)
df.to_csv(f'DeepLearning/Thesis/Setup/Data/{fileName}', index=False)
